# AI_RAG source code

This notebook documents the `src/` implementation of the AI_RAG solution using the **same logical grouping** defined in `infra/docker/make_production_layout.py`.

## How this notebook is organized

The production layout script copies `src/` files into these groups:

- `backend/API_backend`
- `backend/Generation_answer`
- `backend/Retrieval_pipeline`
- `backend/Ingestion_pipeline`
- `frontend`

## Group 1 — `backend/API_backend`

Files mapped here`:

- `src/backend_main.py`
- `src/backend_config.py`
- `src/requirements.txt`


### `src/backend_main.py`

This as the **backend entry point**: it wires together FastAPI, governance, session state, conversation persistence, and the answer orchestrator so the frontend has a stable HTTP API instead of calling retrieval/generation code directly.

**Three most important code snippets**

#### 1. Governance path bootstrap and import (`src/backend_main.py:L22-L32`)
```python
def _ensure_governance_on_path() -> None:
    here = Path(__file__).resolve()
    for parent in here.parents:
        if (parent / "governance" / "governance_gate.py").exists():
            parent_str = str(parent)
            if parent_str not in sys.path:
                sys.path.insert(0, parent_str)
            return

_ensure_governance_on_path()
from governance.governance_gate import get_governance_gate
```

**Why it matters**

The backend imports `governance.governance_gate` only after dynamically finding a parent folder that contains `governance/governance_gate.py`. That is a practical compatibility layer: the same code can run after the `Production/...` packaging step without hardcoding one filesystem layout.

#### 2. API contract models (`src/backend_main.py:L95-L124`)
```python
class ChatRequest(BaseModel):
    session_id: str
    message: str
    policy_type: Optional[str] = None
    policies: Optional[List[Dict]] = None

class GovernanceResult(BaseModel):
    blocked: bool = False
    stage: Optional[str] = None
    input: Optional[Dict[str, Any]] = None
    output: Optional[Dict[str, Any]] = None

class ChatResponse(BaseModel):
    message: str
    needs_clarification: bool
    requires_language_confirmation: bool
    policy_type: Optional[str]
    timings_ms: Optional[Dict[str, float]] = None
    rag_breakdown_ms: Optional[Dict[str, float]] = None
    governance: Optional[GovernanceResult] = None

class GovernanceSettingsUpdate(BaseModel):
    check_input: Optional[bool] = None
    check_output: Optional[bool] = None
    enable_content_safety: Optional[bool] = None
    enable_jailbreak_detection: Optional[bool] = None
    enable_azure_content_safety: Optional[bool] = None
    enable_pii: Optional[bool] = None
    compliance_standards: Optional[List[str]] = None
    azure_severity_threshold: Optional[int] = Field(default=None, ge=0, le=6)
```

**Why it matters**

These Pydantic models define the backend/frontend contract explicitly: session creation, chat input, governance payloads, and governance-settings updates. This is a strong design choice because it makes request validation and response shape enforcement happen at the boundary instead of deep inside business logic.

#### 3. Session creation and chat delegation (`src/backend_main.py:L139-L154, L188-L198`)
```python
def create_session(request: SessionCreateRequest):
    try:
        user_id = request.user_id
        user_profile = cosmos_manager.get_user_profile(user_id)
        if not user_profile:
            raise HTTPException(status_code=404, detail=f"User profile not found: {user_id}")

        language = user_profile.get("language", "en")
        policies = user_profile.get("policies", [])
        if isinstance(policies, dict):
            policies = [{k: v} for k, v in policies.items()]

        session_id = session_manager.create_session(user_id=user_id, language=language)
        #cosmos_manager.create_conversation(session_id=session_id, user_id=user_id, language=language)
        past_sessions = cosmos_manager.get_user_sessions(user_id, limit=10)
        logger.info(f"Created session {session_id} for user {user_id}")
# ...
@app.post("/chat", response_model=ChatResponse)
def chat(request: ChatRequest):
    try:
        response = agent_orchestrator.process_message(
            session_id=request.session_id,
            user_message=request.message,
            policy_type=request.policy_type,
            policies=request.policies
        )
        logger.info(f"chat keys returned: {list(response.keys())}")
        return ChatResponse(**response)
```

**Why it matters**

`create_session()` merges Cosmos profile data with Redis session state, while `chat()` delegates the main answer flow to `agent_orchestrator.process_message(...)`. That separation keeps HTTP concerns in this file and the RAG workflow in the orchestration module.

**Overall script flow**

1. Load config/logger imports from `Production.backend...` modules.
2. Make sure governance imports are resolvable in the packaged layout.
3. Initialize telemetry, then instantiate `SessionManager`, `CosmosManager`, and `AgentOrchestrator`.
4. Build the FastAPI app and CORS middleware.
5. Expose endpoints for session lifecycle, language confirmation, chat, history, user evaluation, and governance settings/audit.
6. Convert internal manager/orchestrator results into stable HTTP responses and wrap failures in `HTTPException`.


### `src/backend_config.py`

This file as the **single configuration surface** for the whole solution. It keeps Azure/OpenAI/Search/Redis/Cosmos wiring in one place so environment changes do not require code edits across many modules.

**Three most important code snippets**

#### 1. Central Azure/runtime configuration (`src/backend_config.py:L25-L35, L47-L58, L63-L80`)
```python
    # Azure ChatGPT configuration
    AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT", "")
    AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY", "")
    AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4.1")
    AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-02-15-preview")

    # Azure OpenAI for embeddings
    AZURE_OPENAI_EMBEDDINGS_ENDPOINT = os.getenv("AZURE_OPENAI_EMBEDDINGS_ENDPOINT", "")
    AZURE_OPENAI_EMBEDDINGS_KEY = os.getenv("AZURE_OPENAI_EMBEDDINGS_KEY", "")
    AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT", "text-embedding-3-small")
    AZURE_OPENAI_EMBEDDINGS_MODEL = os.getenv("AZURE_OPENAI_EMBEDDINGS_MODEL", "text-embedding-3-small")
# ...
    # Redis configuration
    REDIS_HOST = os.getenv("REDIS_HOST", "localhost")
    REDIS_PORT = int(os.getenv("REDIS_PORT", 6379))
    REDIS_PASSWORD = os.getenv("REDIS_PASSWORD", None)
    REDIS_SSL = os.getenv("REDIS_SSL", "True").lower()
    REDIS_DB = int(os.getenv("REDIS_DB", 0))
    REDIS_SESSION_TTL = 1800
    REDIS_PORT = int(os.getenv("REDIS_PORT", 6379))

    # Azure Cosmos DB configuration
    COSMOS_ENDPOINT = os.getenv("COSMOS_ENDPOINT", "")
    COSMOS_KEY = os.getenv("COSMOS_KEY", "")
# ...
    # Azure Search configuration
    AZURE_SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT", "")
    AZURE_SEARCH_API_KEY = os.getenv("AZURE_SEARCH_API_KEY", "")
    AZURE_SEARCH_RECREATE_INDEX = os.getenv("AZURE_SEARCH_RECREATE_INDEX", "False").lower()
    AZURE_SEARCH_INDEX = os.getenv("AZURE_SEARCH_INDEX", "")
    DEFAULT_TOPK = int(os.getenv("DEFAULT_TOPK", 3))
    FINAL_TOP_K = int(os.getenv("FINAL_TOP_K", 3))
    k_nearest_neighbors = int(os.getenv("k_nearest_neighbors", 20))

    # Application configuration
    BACKEND_HOST = os.getenv("BACKEND_HOST", "localhost")
    BACKEND_PORT = int(os.getenv("BACKEND_PORT", 8000))
    BACKEND_URL = os.getenv("BACKEND_URL", f"http://{BACKEND_HOST}:{BACKEND_PORT}")
    FRONTEND_HOST = os.getenv("FRONTEND_HOST", "localhost")
    FRONTEND_PORT = int(os.getenv("FRONTEND_PORT", 8501))
    FRONTEND_URL = os.getenv("FRONTEND_URL", f"http://{FRONTEND_HOST}:{FRONTEND_PORT}")
    COHERE_API_KEY = os.getenv("COHERE_API_KEY")
    BATCH_SIZE = int(os.getenv("BATCH_SIZE", 10))
```

**Why it matters**

`InsuranceConfig` holds the deployment names, API versions, endpoints, ports, TTLs, and search defaults that the rest of the system consumes. This is the backbone that makes the same code reusable across local, Docker, dev, and prod environments.

#### 2. Language-facing prompt dictionaries (`src/backend_config.py:L94-L119`)
```python
    LANGUAGE_CONFIRMATION_PROMPT = {
        "en": "Please confirm your language preference or select a different one to continue.",
        "fr": "Veuillez confirmer votre préférence linguistique ou en sélectionner une autre pour continuer.",
        "de": "Bitte bestätigen Sie Ihre Sprachpräferenz oder wählen Sie eine andere aus, um fortzufahren.",
        "it": "Si prega di confermare la preferenza della lingua o selezionarne un'altra per continuare."
    }
    EVALUATION_PROMPT = {
        "en": "\n\n---\n\n**How would you rate this response?** Please rate from 1 (poor) to 5 (excellent).",
        "fr": "\n\n---\n\n**Comment évalueriez-vous cette réponse?** Veuillez noter de 1 (médiocre) à 5 (excellent).",
        "de": "\n\n---\n\n**Wie würden Sie diese Antwort bewerten?** Bitte bewerten Sie von 1 (schlecht) bis 5 (ausgezeichnet).",
        "it": "\n\n---\n\n**Come valuterebbe questa risposta?** Si prega di valutare da 1 (scarso) a 5 (eccellente)."
    }

    @classmethod
    def get_language_display(cls, lang_code: str) -> str:
        return cls.LANGUAGE_DISPLAY.get(lang_code, lang_code.upper())

    @classmethod
    def get_language_confirmation_prompt(cls, lang_code: str) -> str:
        return cls.LANGUAGE_CONFIRMATION_PROMPT.get(lang_code, cls.LANGUAGE_CONFIRMATION_PROMPT["en"])

    @classmethod
    def get_evaluation_prompt(cls, lang_code: str) -> str:
        return cls.EVALUATION_PROMPT.get(lang_code, cls.EVALUATION_PROMPT["en"])
```

**Why it matters**

The file does more than infrastructure config: it also centralizes cross-module language prompts such as language confirmation and evaluation text. That reduces duplicated strings between frontend and backend-adjacent code.

#### 3. Configuration validation (`src/backend_config.py:L121-L133`)
```python
    @classmethod
    def validate_config(cls) -> bool:
        """Validate that all required configuration is present"""
        required = [
            "AZURE_OPENAI_ENDPOINT", "AZURE_OPENAI_API_KEY",
            "COSMOS_ENDPOINT", "COSMOS_KEY", "COSMOS_DATABASE_NAME"
        ]
        missing = [key for key in required if not getattr(cls, key)]
        if missing:
            logger.error(f"Missing required configuration: {', '.join(missing)}")
            return False
        logger.info("Configuration validated successfully")
        return True
```

**Why it matters**

`validate_config()` is a lightweight guardrail that checks for the minimum required Azure/Cosmos settings before runtime. This is a fail-fast mechanism that catches deployment mistakes early.

**Overall script flow**

1. Load `.env` values once.
2. Materialize them as class attributes on `InsuranceConfig`.
3. Offer small helpers for language display/prompt lookup.
4. Offer a single validation method for critical required settings.


### `src/requirements.txt` (copied into both backend and frontend groups by the layout script)

This is a **shared dependency manifest**, not an executable script. The production layout duplicates the same `src/requirements.txt` into both `backend/API_backend` and `frontend`, which tells me the project is packaged from one Python dependency baseline.

**Fewer than three meaningful items were used here** because a requirements file is declarative rather than algorithmic.

#### 1. Core platform dependencies (`src/requirements.txt:L11-L21`, decoded from UTF-16)
```text
azure-ai-contentsafety==1.0.0
azure-ai-documentintelligence==1.0.2
azure-common==1.1.28
azure-core==1.38.0
azure-core-tracing-opentelemetry==1.0.0b12
azure-cosmos==4.7.0
azure-identity==1.25.2
azure-monitor-opentelemetry==1.8.6
azure-monitor-opentelemetry-exporter==1.0.0b48
azure-search-documents==11.6.0
azure-storage-blob==12.19.1
```

**Why it matters**

This block shows the solution is built around Azure Content Safety, Document Intelligence, Cosmos DB, Azure Monitor, Azure Search, and Blob Storage. That matches the architecture implied by the runtime modules.

#### 2. Orchestration, app, and notebook tooling (`src/requirements.txt:L58-L79`, decoded from UTF-16)
```text
ipykernel==7.2.0
ipython==9.10.0
ipython_pygments_lexers==1.1.1
isodate==0.7.2
jedi==0.19.2
Jinja2==3.1.6
jiter==0.13.0
jsonpatch==1.33
jsonpointer==3.0.0
jsonschema==4.26.0
jsonschema-specifications==2025.9.1
jupyter_client==8.8.0
jupyter_core==5.9.1
langchain==1.2.10
langchain-community==0.4.1
langchain-core==1.2.13
langchain-openai==1.1.9
langchain-text-splitters==1.1.1
langdetect==1.0.9
langgraph==1.0.8
langgraph-checkpoint==4.0.0
langgraph-prebuilt==1.0.7
```

**Why it matters**

This section shows the stack breadth: Jupyter support, LangChain/LangGraph, and OpenAI integrations all sit in the same environment. The repo covers both application runtime and developer analysis workflows.

**Overall file logic**

There is no control flow here; the value is in what it reveals about the system's operational dependencies and shared packaging model.


## Group 2 — `backend/Generation_answer`

Files mapped here:

- `src/agent_orchestrator.py`
- `src/cosmos_manager.py`
- `src/session_manager.py`
- `src/language_manager.py`
- `src/telemetry_context.py`
- `src/app_insights_init.py`


### `src/agent_orchestrator.py`

This is the **core answer-generation coordinator**. It as the module that turns a chat request into a governed RAG response by sequencing validation, retrieval, prompt construction, LLM generation, and conversation persistence.

**Three most important code snippets**

#### 1. State model and LangGraph workflow (`src/agent_orchestrator.py:L112-L161`)
```python
class AgentState(TypedDict):
    session_id: str
    user_id: str
    language: str
    language_confirmed: bool
    policy_type: Optional[str]
    user_message: str
    conversation_history: List[Dict]
    context: Optional[Dict[str, Dict[str, Any]]]
    content: Optional[str]
    response: Optional[str]
    needs_clarification: bool
    policies: Optional[List[Dict]]
    timings_ms: Dict[str, float]
    governance_input: Optional[Dict[str, Any]]
    governance_output: Optional[Dict[str, Any]]
    governance_blocked: bool
    governance_block_stage: Optional[str]

class AgentOrchestrator:
    def __init__(self, session_manager: SessionManager, cosmos_manager: CosmosManager):
        self.session_manager = session_manager
        self.cosmos_manager = cosmos_manager
        self.llm = AzureChatOpenAI(
            azure_endpoint=InsuranceConfig.AZURE_OPENAI_ENDPOINT,
            api_key=InsuranceConfig.AZURE_OPENAI_API_KEY,  # type:ignore
            azure_deployment=InsuranceConfig.AZURE_OPENAI_DEPLOYMENT,
            api_version=InsuranceConfig.AZURE_OPENAI_API_VERSION,
            temperature=InsuranceConfig.TEMPERATURE_POLICY,
        )
        self.governance_gate = get_governance_gate()
        self.workflow = self._build_workflow()
        logger.info("AgentOrchestrator initialized (perf)")

    def _build_workflow(self) -> "CompiledStateGraph[AgentState, None, AgentState, AgentState]":
        workflow = StateGraph(AgentState)
        workflow.add_node("validate_input", self._validate_input)
        workflow.add_node("retrieve_context", self._retrieve_context)
        workflow.add_node("generate_response", self._generate_response)
        workflow.add_node("validate_output", self._validate_output)
        workflow.set_entry_point("validate_input")
        workflow.add_conditional_edges(
            "validate_input",
            self._route_after_input_governance,
            {"retrieve_context": "retrieve_context", "end": END},
        )
        workflow.add_edge("retrieve_context", "generate_response")
        workflow.add_edge("generate_response", "validate_output")
        workflow.add_edge("validate_output", END)
        return workflow.compile()
```

**Why it matters**

`AgentState` makes the workflow state explicit, and `_build_workflow()` wires a deterministic graph: validate input → retrieve context → generate response → validate output. That is a strong engineering choice because it separates steps cleanly and makes governance a first-class part of the answer path.

#### 2. Governance gates around the workflow (`src/agent_orchestrator.py:L163-L190`)
```python
    def _validate_input(self, state: AgentState) -> AgentState:
        result = self.governance_gate.validate_input(state.get("user_message") or "")
        state["governance_input"] = result
        if not result.get("passed", True):
            language = state.get("language") or "en"
            state["governance_blocked"] = True
            state["governance_block_stage"] = "input"
            state["response"] = self.governance_gate.blocked_user_message("input", language)
            logger.warning("Inbound governance blocked session %s: %s", state.get("session_id"), result.get("violations"))
        return state

    def _route_after_input_governance(self, state: AgentState) -> str:
        if state.get("governance_blocked"):
            return "end"
        return "retrieve_context"

    def _validate_output(self, state: AgentState) -> AgentState:
        if state.get("governance_blocked"):
            return state
        result = self.governance_gate.validate_output(state.get("response") or "")
        state["governance_output"] = result
        if not result.get("passed", True):
            language = state.get("language") or "en"
            state["governance_blocked"] = True
            state["governance_block_stage"] = "output"
            state["response"] = self.governance_gate.blocked_user_message("output", language)
            logger.warning("Outbound governance blocked session %s: %s", state.get("session_id"), result.get("violations"))
        return state
```

**Why it matters**

The orchestrator validates both inbound user text and outbound assistant text. That is important because the code treats safety/compliance as part of orchestration, not as an afterthought bolted onto the UI.

#### 3. Retrieval normalization and grounded generation (`src/agent_orchestrator.py:L209-L224, L242-L250, L329-L336, L343-L354`)
```python
                context = retrieve_policy_context(
                    index_name,
                    user_message,
                    filtro=language,  # type: ignore
                )
                rag_total_ms = (time.perf_counter() - t0) * 1000.0
                if not isinstance(context, dict):
                    context = {}
                contents: list[str] = []
                for chunk in context.values():
                    if isinstance(chunk, dict) and "content" in chunk:
                        contents.append(str(chunk["llm_context"]) if chunk.get("llm_context") else str(chunk["content"]))

                write_contents_readable(contents, contents_readable)
                state["content"] = "\n\n".join(contents)
                state["context"] = context
# ...
        context_str = format_context(context)
        current_policies_str = stringify_current_policies(current_policies)
        current_prompt = f"""You are an insurance assistant. Generate a customer-facing MARKDOWN report answering the user's question using
        ONLY the following data
        INPUT: 
        - The user question is: {user_message}
        - The current policies the user has are: {current_policies_str}
        - The context is: {context_str}.
        Do not use external knowledge. Write in {language}. Keep acronyms (LAMal, LCA, etc.) unchanged.
# ...
        system_prompt = new_prompt if policy_type == "new" else current_prompt
        messages = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=state["user_message"]),
        ]
        try:
            with span_perf(
                "llm_chat.generate_response",
# ...
                t0 = time.perf_counter()
                result = llm_chat.invoke(messages)
                state["timings_ms"]["llm_generation_ms"] = (time.perf_counter() - t0) * 1000.0
                response = result.content.strip()  # type: ignore
                state["response"] = response
                print(result.usage_metadata)
                token_answer_query={}
                token_answer_query["type"] = "answer_query" 
                token_answer_query["input_tokens"] = (result.usage_metadata).get("input_tokens", 0)  # type: ignore
                token_answer_query["output_tokens"] = (result.usage_metadata).get("output_tokens", 0)  # type: ignore
                token_answer_query["total_tokens"] = (result.usage_metadata).get("total_tokens", 0)  # type: ignore
                token_chat_usage(token_answer_generation, token_answer_query, state["user_message"])
```

**Why it matters**

These excerpts show the two core generation inputs: normalized retrieval context and a constrained prompt that explicitly says "ONLY the following data." It also records token usage after LLM invocation, which is important for operational visibility.

**Overall script flow**

1. Resolve governance imports for the packaged layout.
2. Create one orchestrator with the chat LLM, the governance gate, and a compiled workflow.
3. Validate inbound user text.
4. Retrieve relevant policy chunks and flatten them for prompting.
5. Generate a structured answer in the active language.
6. Validate outbound content, then persist both user and assistant messages to Cosmos.
7. Return a response payload that also includes governance status and timing metadata.


## How `_build_workflow()` Creates the LangGraph

`_build_workflow()` defines the order in which one chat request is processed. It creates a `StateGraph` whose shared data is `AgentState`, registers functions as workflow nodes, connects those nodes with edges, and compiles the graph into an executable workflow.

```python
workflow = StateGraph(AgentState)
```

This creates an initially empty graph. Every step reads and updates the same state dictionary, which contains values such as the user's message, retrieved context, generated response, timings, and governance results.

### Workflow nodes

```python
workflow.add_node("validate_input", self._validate_input)
workflow.add_node("retrieve_context", self._retrieve_context)
workflow.add_node("generate_response", self._generate_response)
workflow.add_node("validate_output", self._validate_output)
```

Each `add_node()` statement gives a graph step a name and connects it to a method on `AgentOrchestrator`:

| Graph node | Method called | Responsibility |
|---|---|---|
| `validate_input` | `_validate_input` | Check the user's message for governance violations. |
| `retrieve_context` | `_retrieve_context` | Query Azure AI Search and store relevant policy chunks in the state. |
| `generate_response` | `_generate_response` | Build the prompt and ask Azure OpenAI for an answer. |
| `validate_output` | `_validate_output` | Check the generated answer before returning it to the user. |

### Entry point and conditional routing

```python
workflow.set_entry_point("validate_input")
workflow.add_conditional_edges(
    "validate_input",
    self._route_after_input_governance,
    {"retrieve_context": "retrieve_context", "end": END},
)
```

Every execution starts at `validate_input`. After validation, LangGraph calls `_route_after_input_governance(state)`:

```python
def _route_after_input_governance(self, state: AgentState) -> str:
    if state.get("governance_blocked"):
        return "end"
    return "retrieve_context"
```

A blocked request goes directly to `END`, so Azure AI Search and the LLM are not called. A permitted request continues to retrieval.

### Normal-path edges and compilation

```python
workflow.add_edge("retrieve_context", "generate_response")
workflow.add_edge("generate_response", "validate_output")
workflow.add_edge("validate_output", END)
return workflow.compile()
```

These are unconditional edges for an allowed request. `compile()` turns the graph definition into a `CompiledStateGraph`, stored as `self.workflow`. Later, `process_message()` executes it with:

```python
final_state = self.workflow.invoke(initial_state)
```

The returned `final_state` contains all changes made by the executed nodes, including the response, retrieved context, timings, and governance information.

### Graph

```mermaid
flowchart TD
    Start([Chat request]) --> Validate[validate_input\n_validate_input]
    Validate --> Route{governance_blocked?}
    Route -->|Yes| Blocked[Store blocked response\ninput stage]
    Blocked --> End([END])
    Route -->|No| Retrieve[retrieve_context\n_retrieve_context]
    Retrieve --> Generate[generate_response\n_generate_response]
    Generate --> ValidateOutput[validate_output\n_validate_output]
    ValidateOutput --> End
```

The key governance property is that input validation is a decision point: unsafe input ends the graph early, while allowed input follows the full RAG path.

### `src/cosmos_manager.py`

This is the **conversation and profile persistence adapter**. It isolates Azure Cosmos DB access behind business-level methods so the rest of the application never has to deal with container APIs directly.

**Three most important code snippets**

#### 1. Cosmos client/bootstrap (`src/cosmos_manager.py:L25-L37`)
```python
    def __init__(self):
        try:
            self.client = CosmosClient(
                InsuranceConfig.COSMOS_ENDPOINT,
                InsuranceConfig.COSMOS_KEY
            )
            self.database = self.client.get_database_client(InsuranceConfig.COSMOS_DATABASE_NAME)
            self.user_profile_container = self.database.get_container_client(InsuranceConfig.COSMOS_USER_PROFILE_CONTAINER)
            self.chat_history_container = self.database.get_container_client(InsuranceConfig.COSMOS_CHAT_HISTORY_CONTAINER)
            logger.info("Cosmos DB connection established")
        except Exception as e:
            logger.error(f"[Cosmos] Failed to initialize Cosmos DB: {e}", exc_info=True)
            raise
```

**Why it matters**

The constructor binds one database plus separate profile/history containers from config values. That keeps persistence concerns centralized and makes failures visible immediately during app startup.

#### 2. Conversation document creation and message append (`src/cosmos_manager.py:L77-L121`)
```python
    def create_conversation(self, session_id: str, user_id: str, language: str, policy_type: Optional[str] = None) -> bool:
        """Create new conversation document. Returns True if successful, False otherwise."""
        try:
            document = {
                "id": session_id,
                "session_id": session_id,
                "user_id": user_id,
                "language": language,
                "policy_type": policy_type,
                "messages": [],
                "first_message": None,
                "evaluation_score": None,
                "created_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
                "updated_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
            }
            self.chat_history_container.create_item(body=document)
            logger.info(f"Created conversation document for session {session_id}")
            return True
        except Exception as e:
            logger.error(f"Error creating conversation: {e}", exc_info=True)
            return False

    def add_messages(self, session_id: str, messages: List[Dict]) -> bool:
        """Add messages to conversation. Returns True on success, False otherwise."""
        try:
            document = self.chat_history_container.read_item(
                item=session_id,
                partition_key=session_id
            )
            if document.get("first_message") is None:
                for msg in messages:
                    if msg.get("role") == "user":
                        document["first_message"] = msg.get("content")
                        break
            document["messages"].extend(messages)
            document["updated_at"] = datetime.datetime.now(datetime.timezone.utc).isoformat()
            self.chat_history_container.replace_item(item=session_id, body=document)
            logger.info(f"Added {len(messages)} message(s) to session {session_id}")
            return True
        except exceptions.CosmosResourceNotFoundError:
            logger.error(f"Conversation not found: {session_id}")
            return False
        except Exception as e:
            logger.error(f"Error adding messages: {e}", exc_info=True)
            return False
```

**Why it matters**

`create_conversation()` establishes the session document shape, and `add_messages()` appends chat turns while also preserving the first user message as a summary field. That is a sensible design for both chat playback and later session browsing.

#### 3. Past-session summarization (`src/cosmos_manager.py:L165-L204`)
```python
    def get_user_sessions(self, user_id: str, limit: int = 10) -> List[Dict]:
        """Get user's past sessions. Returns empty list on error/not found. Always includes a user query extract."""
        try:
            query = (
                "SELECT c.session_id, c.created_at, c.updated_at, c.policy_type, "
                "c.evaluation_score, c.first_message, c.messages "
                "FROM c WHERE c.user_id = @user_id "
                "ORDER BY c.updated_at DESC OFFSET 0 LIMIT @limit"
            )
            parameters = [
                {"name": "@user_id", "value": user_id},
                {"name": "@limit", "value": limit}
            ]
            items = list(self.chat_history_container.query_items(
                query=query,
                parameters=parameters,
                enable_cross_partition_query=True
            ))
            sessions = []
            for item in items:
                # Prefer first_message, else find first user message in messages
                extract = item.get("first_message")
                if not extract:
                    messages = item.get("messages", [])
                    for msg in messages:
                        if msg.get("role") == "user" and msg.get("content"):
                            extract = msg["content"]
                            break
                sessions.append({
                    "session_id": item["session_id"],
                    "date": item.get("updated_at", item.get("created_at")),
                    "policy_type": item.get("policy_type"),
                    "evaluation_score": item.get("evaluation_score"),
                    "first_message": extract or ""
                })
            logger.info(f"Retrieved {len(sessions)} sessions for user {user_id}")
            return sessions
        except Exception as e:
            logger.error(f"Error retrieving user sessions: {e}", exc_info=True)
            return []
```

**Why it matters**

`get_user_sessions()` turns raw Cosmos items into sidebar-ready session summaries with date, policy type, evaluation score, and a first-message extract. This is the bridge between durable storage and the frontend history UX.

**Overall script flow**

1. Connect to Cosmos once.
2. Expose profile read/update methods.
3. Create per-session conversation documents.
4. Append messages and evaluation scores over time.
5. Return either full chat history or compact session summaries for the UI.


### `src/session_manager.py`

This is the **ephemeral session-state layer**. IIt has separated Redis session metadata from Cosmos chat history so fast-changing UI state does not require a full database write on every interaction.

**Three most important code snippets**

#### 1. Session creation with TTL (`src/session_manager.py:L43-L66`)
```python
    def create_session(self, user_id: str, language: str = "en") -> str:
        """Create new session metadata in Redis; returns session_id or Exception."""
        session_id = str(uuid.uuid4())
        session_data = {
            "session_id": session_id,
            "user_id": user_id,
            "language": language,
            "language_confirmed": False,
            "policy_type": None,
            "created_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
            "last_activity": datetime.datetime.now(datetime.timezone.utc).isoformat()
        }
        session_key = f"session:{session_id}"
        try:
            self.redis_client.setex(
                session_key,
                InsuranceConfig.REDIS_SESSION_TTL,
                json.dumps(session_data)
            )
            logger.info(f"Created session {session_id} for user {user_id}")
            return session_id
        except Exception as e:
            logger.error(f"Failed to create session {session_id}: {e}", exc_info=True)
            raise
```

**Why it matters**

A new UUID-based session is stored with `setex(...)`, so expiry is built into the storage operation. That is a strong operational choice for a chat app because abandoned sessions clean themselves up.

**Example: automatic session expiry**

Assume `InsuranceConfig.REDIS_SESSION_TTL` is `1800`, meaning 30 minutes. When a user starts a chat, the application might create a UUID such as `a1b2c3d4` and call:

```python
redis_client.setex(
    "session:a1b2c3d4",
    1800,
    '{"user_id": "45.19805-1", "language": "en", "policy_type": null}'
)
```

Redis immediately stores the session value and starts its 1,800-second countdown:

```text
Time 0 minutes:  session:a1b2c3d4 exists; TTL is 1800 seconds
Time 10 minutes: session:a1b2c3d4 exists; TTL is about 1200 seconds
Time 30 minutes: Redis expires and deletes session:a1b2c3d4 automatically
```

If the user returns after expiry, `get_session(session_id)` finds no Redis key. The backend treats the session as expired and returns a response such as:

```json
{
  "message": "Session expired. Please refresh and try again."
}
```

No background cleanup job is required. Redis owns the expiry timer and removes inactive session metadata automatically. The durable conversation history in Cosmos DB is separate, so deleting this temporary Redis key does not necessarily delete the saved chat history.

#### 2. Safe session update (`src/session_manager.py:L81-L100`)
```python
    def update_session(self, session_id: str, **kwargs) -> bool:
        """Update session metadata fields (safe). Returns True on success, False otherwise."""
        session_data = self.get_session(session_id)
        if not session_data:
            logger.error(f"Cannot update non-existent session {session_id}")
            return False
        session_data.update(kwargs)
        session_data["last_activity"] = datetime.datetime.now(datetime.timezone.utc).isoformat()
        session_key = f"session:{session_id}"
        try:
            self.redis_client.setex(
                session_key,
                InsuranceConfig.REDIS_SESSION_TTL,
                json.dumps(session_data)
            )
            logger.info(f"Updated session {session_id}")
            return True
        except Exception as e:
            logger.error(f"Failed to update session {session_id}: {e}", exc_info=True)
            return False
```

**Why it matters**

`update_session()` reloads the current JSON, patches fields, refreshes `last_activity`, and writes the value back with the same TTL. This keeps Redis as the single source of transient session truth.

#### 3. Pending-query persistence (`src/session_manager.py:L128-L146`)
```python
    def update_pending_query(self, session_id: str, pending_query: Optional[str]) -> bool:
        """Update pending query field atomically. Returns True if updated."""
        try:
            session_data = self.get_session(session_id)
            if not session_data:
                logger.error(f"Session {session_id} not found")
                return False
            session_data["pending_query"] = pending_query
            session_key = f"session:{session_id}"
            self.redis_client.setex(
                session_key,
                InsuranceConfig.REDIS_SESSION_TTL,
                json.dumps(session_data)
            )
            logger.info(f"Updated pending_query for session {session_id}: {pending_query}")
            return True
        except Exception as e:
            logger.error(f"Failed to update pending_query: {e}", exc_info=True)
            return False
```

**Why it matters**

The `pending_query` field is important to the UX: if the user asks a question before confirming language or before choosing a policy context, the app can hold the request and replay it later instead of losing it.

**Overall script flow**

1. Connect to Redis.
2. Create JSON session objects with TTL.
3. Read/update/delete/refresh them safely.
4. Carry special transient state such as `pending_query`, language confirmation, and policy type.


### `src/language_manager.py`

This is the **multilingual copy layer** for the product. It keeps UX strings out of the Streamlit logic so language behavior can evolve without scattering text across the UI.

**Three most important code snippets**

#### 1. Language selection and confirmation prompts (`src/language_manager.py:L22-L41`)
```python
    SELECT_LANGUAGE_PROMPT = {
        "en": "Please select your preferred language before asking a question:",
        "fr": "Veuillez sélectionner votre langue préférée avant de poser une question :",
        "de": "Bitte wählen Sie Ihre bevorzugte Sprache, bevor Sie eine Frage stellen:",
        "it": "Si prega di selezionare la lingua preferita prima di fare una domanda:"
    }

    LANGUAGE_SELECTION = {
        "en": "Please select your preferred language from the sidebar to continue.",
        "fr": "Veuillez sélectionner votre langue préférée dans la barre latérale pour continuer.",
        "de": "Bitte wählen Sie Ihre bevorzugte Sprache in der Seitenleiste aus, um fortzufahren.",
        "it": "Si prega di selezionare la lingua preferita dalla barra laterale per continuare."
    }

    LANGUAGE_CONFIRMATION_PROMPT = {
        "en": "🌐 Please confirm your language preference or select another language to continue.",
        "fr": "🌐 Veuillez confirmer votre préférence linguistique ou sélectionner une autre langue pour continuer.",
        "de": "🌐 Bitte bestätigen Sie Ihre Sprachpräferenz oder wählen Sie eine andere Sprache aus, um fortzufahren.",
        "it": "🌐 Si prega di confermare la preferenza della lingua o selezionarne un'altra per continuare."
    }
```

**Why it matters**

These dictionaries define the first critical branching point in the user journey: language selection and confirmation. Keeping them centralized makes the onboarding flow predictable across English, French, German, and Italian.

#### 2. Policy-type button labels (`src/language_manager.py:L64-L77`)
```python
    CATEGORY_BUTTONS = {
        "current": {
            "en": "📋 Current Policies",
            "fr": "📋 Polices Actuelles",
            "de": "📋 Aktuelle Policen",
            "it": "📋 Polizze Attuali"
        },
        "new": {
            "en": "🆕 New Policies",
            "fr": "🆕 Nouvelles Polices",
            "de": "🆕 Neue Policen",
            "it": "🆕 Nuove Polizze"
        }
    }
```

**Why it matters**

The "current" versus "new" policy distinction is a core product concept, and this dictionary ensures that distinction is expressed consistently in every supported language.

#### 3. Generic message lookup (`src/language_manager.py:L378-L388, L403-L434`)
```python
    def get_message(self, key: str, language: str = "en") -> str:
        message_map = {
            "select_language_prompt": self.SELECT_LANGUAGE_PROMPT,
            "language_selection": self.LANGUAGE_SELECTION,
            "category_selection": self.CATEGORY_SELECTION,
            "clarification": self.CLARIFICATION,
            "ask_below": self.ASK_BELOW,
            "configuration": self.CONFIGURATION,
            "change_language": self.CHANGE_LANGUAGE,
            "language_confirmation": self.LANGUAGE_CONFIRMATION_PROMPT,
            "confirm_language_button": self.CONFIRM_LANGUAGE_BUTTON,
# ...
            "processing_query": self.PROCESSING_QUERY,
            "selected_policy_type": self.SELECTED_POLICY_TYPE,
            "language_section": self.LANGUAGE_SECTION,
            "select_language": self.SELECT_LANGUAGE,
            "chat_disabled_caption": self.CHAT_DISABLED_CAPTION,
            "backend_processing_error": self.BACKEND_PROCESSING_ERROR,
            "thinking": self.THINKING,
            "backend_failed_response": self.BACKEND_FAILED_RESPONSE,
            "fatal_error_frontend": self.FATAL_ERROR_FRONTEND,
            "app_title": self.APP_TITLE,
            "logout": self.LOGOUT,
            "governance_section": self.GOVERNANCE_SECTION,
            "governance_help": self.GOVERNANCE_HELP,
            "governance_check_input": self.GOVERNANCE_CHECK_INPUT,
            "governance_check_output": self.GOVERNANCE_CHECK_OUTPUT,
            "governance_content_safety": self.GOVERNANCE_CONTENT_SAFETY,
            "governance_jailbreak": self.GOVERNANCE_JAILBREAK,
            "governance_azure_cs": self.GOVERNANCE_AZURE_CS,
            "governance_pii": self.GOVERNANCE_PII,
            "governance_save": self.GOVERNANCE_SAVE,
            "governance_saved": self.GOVERNANCE_SAVED,
            "governance_azure_ok": self.GOVERNANCE_AZURE_OK,
            "governance_azure_missing": self.GOVERNANCE_AZURE_MISSING,
            "governance_blocked": self.GOVERNANCE_BLOCKED,
            "governance_passed": self.GOVERNANCE_PASSED,
            "governance_load_error": self.GOVERNANCE_LOAD_ERROR,
        }
        messages = message_map.get(key, {})
        return messages.get(language, messages.get("en", ""))

    def get_category_button_label(self, category: str, language: str = "en") -> str:
        return self.CATEGORY_BUTTONS.get(category, {}).get(language, category)
```

**Why it matters**

`get_message()` and `get_category_button_label()` turn a large set of static dictionaries into one small runtime API. This is the key abstraction that keeps the frontend code from becoming a maze of per-language `if` statements.

**Overall script flow**

There is almost no algorithmic flow; the module is mostly curated multilingual data plus two lookup helpers. Its value is architectural: it decouples product copy from UI control logic.


### `src/telemetry_context.py`

This is a **tiny resilience wrapper** around OpenTelemetry spans.

#### 1. `span_perf(...)` no-op-safe telemetry context (`src/telemetry_context.py:L19-L47`)
```python
try:
    from opentelemetry import trace
    OPENTELEMETRY_OK = True
except ImportError:
    logger.warning("[Telemetry] opentelemetry not installed, telemetry spans will be disabled.")
    OPENTELEMETRY_OK = False

@contextmanager
def span_perf(name: str, attrs: Optional[Dict[str, Any]] = None) -> Iterator[None]:
    """
    Context manager for OpenTelemetry perf spans, safe for missing telemetry dependencies.
    If OpenTelemetry is unavailable, this is a no-op (but logs the usage).
    """
    if OPENTELEMETRY_OK:
        try:
            tracer = trace.get_tracer(__name__)
            with tracer.start_as_current_span(name) as span:
                if attrs:
                    for k, v in attrs.items():
                        if v is None:
                            continue
                        span.set_attribute(k, v)
                yield
        except Exception as e:
            logger.error(f"[Telemetry] Failed to create telemetry span '{name}': {e}", exc_info=True)
            yield
    else:
        logger.info(f"[Telemetry] NO-OP: span_perf called without OpenTelemetry: {name}")
        yield
```

**Why it matters**

The module first detects whether `opentelemetry` is installed, then `span_perf(...)` either creates a real span or degrades into a logged no-op. That is good defensive engineering: observability can enrich the system, but it should not break the chat path.

**Overall script flow**

Import OpenTelemetry if available, expose one context manager, and make telemetry optional instead of mandatory.


### `src/app_insights_init.py`

This is the **one-time Application Insights/OpenTelemetry bootstrap** for the backend service.

#### 1. `configure_app_insights_perf(...)` (`src/app_insights_init.py:L19-L47`)
```python
def configure_app_insights_perf(*, service_name: str) -> bool:
    """
    Configure OpenTelemetry -> Azure Monitor exporter.
    Returns True if configured, False if missing env or failure.
    Logs all exceptions (never throws).
    """
    conn_str: Optional[str] = os.getenv("APPLICATIONINSIGHTS_CONNECTION_STRING")
    if not conn_str or not conn_str.strip():
        _logger.warning("No APPLICATIONINSIGHTS_CONNECTION_STRING found. Skipping App Insights perf config.")
        return False

    try:
        from azure.monitor.opentelemetry import configure_azure_monitor
        from opentelemetry import trace
        from opentelemetry.sdk.resources import Resource
        from opentelemetry.sdk.trace import TracerProvider

        resource = Resource.create({
            "service.name": service_name,
        })
        provider = TracerProvider(resource=resource)
        trace.set_tracer_provider(provider)

        configure_azure_monitor()
        _logger.info(f"Application Insights/OpenTelemetry initialized for service: {service_name}")
        return True
    except Exception as e:
        _logger.error(f"App Insights/OpenTelemetry configuration failed: {e}", exc_info=True)
        return False
```

**Why it matters**

This function checks for `APPLICATIONINSIGHTS_CONNECTION_STRING`, builds an OpenTelemetry tracer provider, and calls `configure_azure_monitor()`. Keep observability setup isolated, optional, and failure-tolerant.

**Overall script flow**

Read the connection string, exit early if it is missing, otherwise initialize Azure Monitor tracing and return a boolean success flag.


## Group 3 — `backend/Retrieval_pipeline`

Files mapped here:

- `src/rag_retriever.py`
- `src/evaluation_llm_answer.py`
- `src/rrf_retrieval_debug_logging.py`


### `src/rag_retriever.py`

This module is the **retrieval engine** for the chatbot. It owns the Azure Search call pattern, the reciprocal-rank-fusion merge logic, and the final chunk selection that feeds answer generation.

**Three most important code snippets**

#### 1. Robust score extraction helpers (`src/rag_retriever.py:L47-L56`)
```python
def _get_reranker_score(hit: Dict[str, Any]) -> float:
    """Try all keys in Azure Search reranker score to be robust."""
    for key in ["@search.reranker_score", "@search.rerankerScore", "@search.score"]:
        v = hit.get(key)
        if isinstance(v, (int, float)):
            return float(v)
    return 0.0

def _rrf_increment(rank_index_zero_based: int, k: int = 60) -> float:
    return 1.0 / (k + rank_index_zero_based + 1)
```

**How `_get_reranker_score(...)` works**

A `hit` is one chunk returned by Azure AI Search. It includes indexed document fields, such as `chunk_id`, `content`, and `title`, plus search-generated ranking fields. For example:

```python
hit = {
    "chunk_id": "policy-42",
    "title": "Diversa Confort",
    "content": "Spa treatments are excluded.",
    "@search.reranker_score": 3.82,
}
```

The function returns `3.82`. It checks alternative score keys in a fixed priority order because Azure Search SDK response representations can vary:

1. `@search.reranker_score`
2. `@search.rerankerScore`
3. `@search.score` as a fallback when no semantic reranker score is available

`hit.get(key)` safely returns `None` when a key is absent. The `isinstance(v, (int, float))` check accepts only numeric values, and `float(v)` gives callers a consistent return type. If no numeric score exists, the function returns `0.0` rather than failing.

The helper is used to sort each query's hits from highest semantic relevance to lowest:

```python
hits_sorted = sorted(
    hits,
    key=lambda h: (-_get_reranker_score(h), str(h.get("chunk_id", ""))),
)
```

The leading minus sign sorts larger scores first. `chunk_id` is a stable tie-breaker, so equal scores produce a deterministic order. The helper is also used to retain the strongest version of a chunk that appears in multiple searches and to record debugging information.

**Reranker score versus RRF score**

The reranker score is Azure AI Search's semantic relevance score for one chunk within one query result list. The RRF score is this application's score for combining ranks across query variants. The RRF contribution from a result at zero-based rank $r$ is:

$$
\operatorname{RRF}(r) = \frac{1}{k + r + 1}
$$

where $k$ is normally `60`. The reranker score determines ordering within a result list; RRF combines those rank positions when several expanded queries are used.

**Why it matters**

The retriever tolerates different score field names coming back from Azure Search and normalizes RRF increments in one place. Small helpers like these are important because ranking code becomes fragile quickly if provider payloads vary.

#### 2. Hybrid Azure AI Search, merge, and selective compression (`src/rag_retriever.py:L98-L113, L119-L129, L131-L136, L163-L176`)
```python
            try:
                vector_query = VectorizedQuery(
                    vector=exp_query_vector,
                    k_nearest_neighbors=k_nearest_neighbors,
                    fields=vector_field,
                )
                results_iter = search_client.search(
                    search_text=exp_query,
                    vector_queries=[vector_query],
                    select=select_fields,
                    top=top_per_query,
                    query_type="semantic",
                    semantic_configuration_name=semantic_configuration_name,
                    query_caption="extractive",
                    filter=fil,
                )
# ...
            hits_sorted = sorted(
                hits,
                key=lambda h: (-_get_reranker_score(h), str(h.get("chunk_id", ""))),
            )
            per_query_hits_sorted.append(hits_sorted)

            for rank, hit in enumerate(hits_sorted):
                chunk_id = hit.get("chunk_id")
                if not chunk_id:
                    logger.warning("[RRF] Hit missing chunk_id--skipped for RRF.")
                    continue
# ...
                reranker_score = _get_reranker_score(hit)
                existing = best_hit_by_chunk.get(chunk_id)
                if existing is None or reranker_score > _get_reranker_score(existing):
                    best_hit_by_chunk[chunk_id] = hit
                rrf_score_by_chunk[chunk_id] = rrf_score_by_chunk.get(chunk_id, 0.0) + _rrf_increment(rank, k=rrf_k)
                debug_ranks.setdefault(chunk_id, []).append((q_idx, rank + 1, reranker_score))
# ...
            # Logic: Only compress 'text' AND if it's large (>= 800 chars)
            if chunk_type == "text" and content_length >= 800 and captions:
                # Use the extractive caption (compressed version)
                if isinstance(captions, list) and captions:
                    hit_chunk["llm_context"] = getattr(captions[0], "text", "")
                else:
                    hit_chunk["llm_context"] = ""

                hit_chunk["is_compressed"] = True
                logger.info(f"[RRF] Compressed chunk {chunk_id} ({content_length} chars -> caption)")
            else:
                # Keep full content for tables, images, or small text chunks
                hit_chunk["llm_context"] = content_body
                hit_chunk["is_compressed"] = False
```

**Why both `search_text` and `VectorizedQuery` are sent**

This is a **hybrid search** request. It sends the same question in two different representations because text search and vector search have different strengths.

```python
search_text=exp_query
```

This sends the human-readable question for lexical matching. It is especially useful for exact policy names, acronyms, product codes, numbers, and terms that must match precisely, such as `LAMal`, `CHF 500`, or `Diversa Confort`.

```python
exp_query_vector = get_embedding(exp_query, embeddings)
vector_query = VectorizedQuery(
    vector=exp_query_vector,
    k_nearest_neighbors=k_nearest_neighbors,
    fields="content_vector",
)
```

`get_embedding(...)` converts the question into a numerical representation of its meaning. `VectorizedQuery` tells Azure AI Search to compare that vector against vectors stored in the `content_vector` field and retrieve the nearest chunks.

For example, a user might ask:

```text
Does Diversa Confort cover spa treatments?
```

A document may instead say:

```text
Wellness services and thermal cures are excluded.
```

Text search may miss or lower-rank this result because the wording differs. Vector search can recognize that spa treatments, wellness services, and thermal cures are related concepts.

| Search type | Strength | Limitation |
|---|---|---|
| Text search | Exact terms, product names, codes, and numbers | Can miss synonyms or paraphrases |
| Vector search | Meaning, synonyms, and different phrasing | Can return a related but incorrect policy |
| Hybrid search | Combines both signals | Needs retrieval tuning |

`vector_queries` is a list because Azure AI Search supports multiple vector queries in one request. This implementation currently sends a list containing one vector query:

```python
vector_queries=[vector_query]
```

The additional semantic settings ask Azure AI Search to rerank the combined candidate set and return extractive captions:

```python
query_type="semantic"
semantic_configuration_name="insurance-semantic-config"
query_caption="extractive"
```

**Current behavior note**

The query-expansion code in `retrieve_policy_context(...)` is commented out. The live code sets:

```python
expanded_queries = [query]
```

Therefore, RRF currently processes only one query result list. It still ranks and selects chunks, but the cross-query fusion benefit appears only if multiple expanded queries are enabled.

**Why it matters**

This excerpt captures the retriever's main design: language-filtered hybrid search, semantic reranking, RRF-based chunk merging, and compression only for long text chunks with extractive captions. That is a precise, defensible trade-off between exact matching, semantic recall, and prompt size.

#### 3. End-to-end retrieval entry point (`src/rag_retriever.py:L243-L247, L294-L309`)
```python
    try:
        search_client = SearchClient(
            endpoint=InsuranceConfig.AZURE_SEARCH_ENDPOINT,  # type: ignore
            index_name=index_name,
            credential=AzureKeyCredential(InsuranceConfig.AZURE_SEARCH_API_KEY)  # type: ignore
# ...
        expanded_queries= [query]

        unique_chunks = search_compression_rrf(
            search_client=search_client,
            expanded_queries=expanded_queries,
            get_embedding=get_embedding,
            embeddings=embeddings,
            top_per_query=top_k,
            k_nearest_neighbors=k_nearest_neighbors,
            language_filter=filtro.upper() if filtro else "",
            vector_field="content_vector",
            semantic_configuration_name="insurance-semantic-config",
            final_top_k=InsuranceConfig.FINAL_TOP_K,
            rrf_k=60,
            select_fields=None
        )
```

**Why it matters**

`retrieve_policy_context()` creates the `SearchClient`, prepares the expanded-query list, and calls `search_compression_rrf(...)`. A notable code-grounded detail: the LLM-based query-expansion block exists but is currently commented out, and the live path explicitly sets `expanded_queries = [query]`.

**Overall script flow**

1. Build embeddings/search clients from config.
2. Optionally prepare multiple query variants (the helper code exists, but the current live branch uses only the original query).
3. Retrieve hybrid-search hits from Azure Search, then semantically rerank them.
4. Merge and compress them for LLM consumption.
5. Return a chunk dictionary keyed by `chunk_id`.


### `src/evaluation_llm_answer.py`

This is the **answer-quality evaluation adapter**. It turns one question, one retrieved context, and one answer into measurable relevance/faithfulness signals using DeepEval on top of Azure OpenAI.

**Three most important code snippets**

#### 1. Nested DeepEval model adapter (`src/evaluation_llm_answer.py:L48-L78`)
```python
    class CustomAzureOpenAI(DeepEvalBaseLLM):
        def __init__(self, model):
            self.model = model
            self.last_response = None

        def load_model(self):
            return self.model

        def generate(self, prompt: str) -> str:
            try:
                self.last_response = self.load_model().invoke(prompt).content
                return self.last_response.content
            except Exception as e:
                logger.error(f"[DeepEval] Failed sync LLM generate: {e}", exc_info=True)
                return "LLM Generation Error"

        async def a_generate(self, prompt: str) -> str:
            try:
                self.last_response = await self.load_model().ainvoke(prompt)
                return self.last_response.content
            except Exception as e:
                logger.error(f"[DeepEval] Failed async LLM generate: {e}", exc_info=True)
                return "LLM Generation Error"

        def get_model_name(self):
            return "Custom Azure OpenAI Model"

        def get_last_response(self):
            if self.last_response and hasattr(self.last_response, "usage_metadata"):
                return self.last_response.usage_metadata
            return None
```

The three metrics assess different things:

**Answer Relevancy**: ```Does the generated answer respond to the user’s question?```

**Faithfulness**: ```Are the answer’s claims supported by the retrieved context?```

**Contextual Relevancy**: ```Are the chunks retrieved from Azure AI Search relevant to the user’s question?```


**Why it matters**

`CustomAzureOpenAI` adapts LangChain's Azure chat model to the `DeepEvalBaseLLM` interface and preserves `usage_metadata`. That is the bridge that lets the evaluation framework reuse the same Azure-hosted model stack as the runtime app.

#### 2. Metric and test-case setup (`src/evaluation_llm_answer.py:L79-L110`)
```python
    try:
        llm_custom_model = AzureChatOpenAI(
            azure_endpoint=InsuranceConfig.AZURE_OPENAI_ENDPOINT,
            api_key=InsuranceConfig.AZURE_OPENAI_API_KEY,  # type:ignore
            azure_deployment=InsuranceConfig.AZURE_OPENAI_DEPLOYMENT,  # type:ignore
            api_version=InsuranceConfig.AZURE_OPENAI_API_VERSION,
            temperature=0.2
        )

        azure_openai = CustomAzureOpenAI(model=llm_custom_model)

        answer_relevancy = AnswerRelevancyMetric(
            threshold=0.7, model=azure_openai, include_reason=True
        )
        faithfulness = FaithfulnessMetric(
            threshold=0.7, model=azure_openai, include_reason=True
        )
        contextual_relevancy = ContextualRelevancyMetric(
            threshold=0.7, model=azure_openai, include_reason=True
        )

        retrieved_context_list = retrieved_context.split("\n\n")
        test_case = LLMTestCase(
            input=test_query,
            actual_output=output,
            retrieval_context=retrieved_context_list
        )

        results = evaluate(
            test_cases=[test_case],
            metrics=[answer_relevancy, faithfulness, contextual_relevancy]
        )
```

**Why it matters**

This block defines three concrete quality dimensions — answer relevancy, faithfulness, and contextual relevancy — and evaluates them against a structured `LLMTestCase`. It makes evaluation explicit instead of subjective.

#### 3. Metric extraction and token logging (`src/evaluation_llm_answer.py:L112-L145`)
```python
        usage_metadata = azure_openai.get_last_response()
        token_evaluation={}

        if usage_metadata:       
            token_evaluation["type"] = "evaluation_llm_response" 
            token_evaluation["input_tokens"] = usage_metadata.get("input_tokens")  # type: ignore
            token_evaluation["output_tokens"] = usage_metadata.get("output_tokens", 0)  # type: ignore
            token_evaluation["total_tokens"] = usage_metadata.get("total_tokens", 0)  # type: ignore
            token_chat_usage(token_evaluation_path, token_evaluation, test_query)

        metrics_dict = {}
        if results.test_results and results.test_results[0].metrics_data:
            for metric_data in results.test_results[0].metrics_data:
                metrics_dict[metric_data.name] = {
                    'score': metric_data.score,
                    'reasoning': metric_data.reason,
                    'success': metric_data.success,
                    'threshold': metric_data.threshold
                }

        metrics_log_path = os.path.join(script_dir, "metrics_results_readable.txt")
        try:
            with open(metrics_log_path, "a", encoding="utf-8") as f:
                f.write(json.dumps(metrics_dict, ensure_ascii=False, indent=2))
                f.write("\n" + "="*60 + "\n")  # Separator between entries
        except Exception as e:
            logger.warning(f"Failed to write metrics_dict to file: {e}")

        return metrics_dict

    except Exception as e:
        logger.error(f"[Eval] Exception in evaluate_response: {e}", exc_info=True)
        # Reliability: fallback to empty dict on error, could optionally raise.
        return {}
```

**Why it matters**

The function records token usage and flattens DeepEval outputs into a serializable dictionary of `score`, `reasoning`, `success`, and `threshold`. That makes the result usable by downstream reporting or manual review.

**Overall script flow**

Wrap Azure Chat OpenAI in a DeepEval-compatible class, run three metrics on one retrieval-backed answer, persist readable results, and return a machine-friendly summary dict.


### `src/rrf_retrieval_debug_logging.py`

This is a **diagnostics helper** for the retriever.

#### 1. Structured RRF debug output (`src/rrf_retrieval_debug_logging.py:L26-L43, L46-L70`)
```python
    logging.info("========== [RRF MERGE DEBUG] ==========")
    logging.info("Total queries: %d", len(expanded_queries))
    logging.info("Total elapsed: %.1f ms", elapsed_ms_total)

    # --- Per-query top results snapshot ---
    for q_idx, q in enumerate(expanded_queries):
        hits = per_query_hits_sorted[q_idx] if q_idx < len(per_query_hits_sorted) else []
        logging.info("---- Query[%d]: %s", q_idx, q)
        if not hits:
            logging.info("  (no hits)")
            continue
        for rank, h in enumerate(hits[:10], start=1):
            chunk_id = str(h.get("chunk_id", ""))
            src = h.get("source_file")
            page = h.get("page_number")
            rr = h.get("@search.reranker_score", h.get("@search.rerankerScore", h.get("@search.score", 0)))
            logging.info("  %02d) chunk_id=%s reranker=%.4f source=%s page=%s",
                         rank, chunk_id, float(rr or 0), src, page)
# ...
    logging.info("---- Merged (RRF) top (first 20) ----")
    for i, (chunk_id, rrf_score) in enumerate(merged_ranked[:20], start=1):
        h = best_hit_by_chunk.get(chunk_id, {})
        src = h.get("source_file")
        page = h.get("page_number")
        best_rr = h.get("@search.reranker_score", h.get("@search.rerankerScore", h.get("@search.score", 0)))
        logging.info("  %02d) chunk_id=%s rrf=%.6f best_reranker=%.4f source=%s page=%s",
                     i, chunk_id, rrf_score, float(best_rr or 0), src, page)

    # --- Final selection (post tie-rule) ---
    logging.info("---- Selected chunk_ids (count=%d) ----", len(selected_ids))
    for chunk_id in selected_ids:
        h = best_hit_by_chunk.get(chunk_id, {})
        src = h.get("source_file")
        page = h.get("page_number")
        best_rr = h.get("@search.reranker_score", h.get("@search.rerankerScore", h.get("@search.score", 0)))
        ranks = debug_ranks.get(chunk_id, [])
        ranks_str = ", ".join([f"q{q_idx}:r{rank}@{rr:.3f}" for (q_idx, rank, rr) in ranks]) or "n/a"
        logging.info("  chunk_id=%s rrf=%.6f best_reranker=%.4f source=%s page=%s ranks=[%s]",
                     chunk_id,
                     rrf_score_by_chunk.get(chunk_id, 0.0),
                     float(best_rr or 0),
                     src,
                     page,
                     ranks_str)
```

**Why it matters**

The function prints three ranking views: per-query hits, merged RRF ranking, and final selected chunks. That is exactly the information you need when a retrieval result "looks wrong" and you need to explain whether the issue came from Azure Search or from the merge policy.

#### 2. Lightweight retrieval timer (`src/rrf_retrieval_debug_logging.py:L75-L89`)
```python
class RrfDebugTimer:
    """Tiny helper to measure elapsed time in ms."""
    def __init__(self) -> None:
        self._t0 = 0.0

    def __enter__(self) -> "RrfDebugTimer":
        self._t0 = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc, tb) -> None:
        pass

    @property
    def elapsed_ms(self) -> float:
        return (time.perf_counter() - self._t0) * 1000.0
```

**Why it matters**

`RrfDebugTimer` is intentionally minimal, but it gives the retriever a cheap elapsed-time measurement that can be included in debug output without entangling the ranking code with timing logic.

**Overall script flow**

Start a timer, run retrieval elsewhere, then log rank-by-rank evidence for how the merged result was produced.


## Group 4 — `backend/Ingestion_pipeline`

Files mapped here:

- `src/indexing_rag.py`
- `src/document_intelligence_rag.py`
- `src/initialization_rag.py`


### `src/indexing_rag.py`

This file is the **search-index construction layer**. It turns cleaned document chunks into Azure AI Search records with embeddings, metadata, vector fields, and semantic-search configuration.

**Three most important code snippets**

#### 1. PDF text cleaning (`src/indexing_rag.py:L146-L171, L173-L182, L185-L197`)
```python
    title = None
    if hasattr(page, "paragraphs") and page.paragraphs:
        for para in page.paragraphs:
            if hasattr(para, "role") and para.role.lower() in ["title", "heading", "header"]:
                title = para.content.strip()
                break
            if hasattr(para, "style") and "heading" in para.style.lower():
                title = para.content.strip()
                break
    if not title and hasattr(page, "lines") and page.lines:
        for line in page.lines:
            line_stripped = line.content.strip()
            if len(line_stripped) > 1 and len(line_stripped.split()) <= 8:
                title = line_stripped
                break

    text_lines = [line.content for line in (page.lines or [])]
    text = '\n'.join(text_lines)
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        if not header_pattern.match(line.strip()) and not footer_pattern.match(line) and not chapter_pattern.match(line):
            cleaned_lines.append(line)
    text_cleaned = '\n'.join(cleaned_lines).strip()
    text_cleaned = re.sub(r'([^\n])\n([^\n])', r'\1 \2', text_cleaned)
    text_cleaned = re.sub(r'\s+', ' ', text_cleaned).strip()
# ...
    edition = None
    if pdf_bytes:
        try:
            pdf_reader = PdfReader(BytesIO(pdf_bytes))
            metadata = pdf_reader.metadata
            if metadata:
                for key, value in metadata.items():
                    if "edition" in str(key).lower():
                        edition = str(value).strip()
                        break
# ...
    if not edition:
        if hasattr(page, "paragraphs") and page.paragraphs:
            for para in page.paragraphs:
                if "edition" in para.content.lower():
                    edition = para.content.strip()
                    break
        if not edition and hasattr(page, "lines") and page.lines:
            for line in page.lines:
                if "edition" in line.content.lower():
                    edition = line.content.strip()
                    break
    edition = edition or None
    return text_cleaned, title, edition
```

**Why it matters**

`clean_pdf_text(...)` removes obvious report noise such as headers/footers, tries to infer a title, and extracts an edition marker when possible. This matters because bad chunk text creates bad embeddings and therefore bad retrieval.

#### 2. Azure Search index schema (`src/indexing_rag.py:L241-L263, L283-L301`)
```python
    fields = [
        SimpleField(name="id", type=SearchFieldDataType.String, key=True, retrievable=True),
        SimpleField(name="chunk_id", type=SearchFieldDataType.String, retrievable=True, filterable=True),
        SearchableField(name="content", type=SearchFieldDataType.String, searchable=True, retrievable=True),
        SearchableField(name="title", type=SearchFieldDataType.String, searchable=True, retrievable=True, sortable=True),
        SearchableField(name="source_file", type=SearchFieldDataType.String, searchable=True, retrievable=True, sortable=True, facetable=True, filterable=True),
        SimpleField(name="page_number", type=SearchFieldDataType.Int32, retrievable=True, filterable=True, sortable=True),
        SearchableField(name="language", type=SearchFieldDataType.String, searchable=False, retrievable=True, filterable=True, facetable=True),
        SearchableField(name="document_type", type=SearchFieldDataType.String, searchable=False, retrievable=True, filterable=True, facetable=True),
        SimpleField(name="total_chunks", type=SearchFieldDataType.Int32, retrievable=True),
        SimpleField(name="token_count", type=SearchFieldDataType.Int32, retrievable=True, filterable=True),
        SimpleField(name="edition_date", type=SearchFieldDataType.String, retrievable=True, filterable=True, sortable=True),
        SearchableField(name="images_tables", type=SearchFieldDataType.String, searchable=True, retrievable=True, sortable=True),
        SearchableField(name="url", type=SearchFieldDataType.String, searchable=False, retrievable=True, sortable=True),
        SimpleField(name="metadata", type=SearchFieldDataType.String, retrievable=True),
        SearchField(
            name="content_vector",
            type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
            searchable=True,
            vector_search_dimensions=1536,
            vector_search_profile_name="insurance-vector-profile"
        )
    ]
# ...
    semantic_config = SemanticConfiguration(
        name="insurance-semantic-config",
        prioritized_fields=SemanticPrioritizedFields(
            title_field=SemanticField(field_name="title"),
            content_fields=[SemanticField(field_name="content")],
            keywords_fields=[SemanticField(field_name="document_type")],
        )
    )
    semantic_search = SemanticSearch(configurations=[semantic_config])
    index = SearchIndex(
        name=index_name,
        fields=fields,
        vector_search=vector_search,
        semantic_search=semantic_search
    )
    try:
        result = index_client.create_or_update_index(index)
        logger.info(f"Index '{result.name}' created/updated successfully.")
        return result
```

**Why it matters**

This snippet defines the actual retrieval substrate: searchable content/title fields, filters for language and document type, semantic configuration, and a 1536-dimensional `content_vector`. This is where the retrieval system is physically represented.

##### Vector and semantic search design principles

> **Compatibility first:** `vector_search_dimensions=1536` must match the length of every embedding written to `content_vector`. This project configures `text-embedding-3-small`, whose default embedding length is 1536. Changing the embedding model or its configured dimensions requires a matching index schema and reindexing existing chunks.

The schema deliberately combines two retrieval mechanisms rather than treating vector search as a replacement for text search:

```text
Question -> keyword/text matching on title and content
         -> vector nearest-neighbor search on content_vector
         -> semantic ranking using title, content, and document_type
```

This supports insurance questions that mix exact terms, such as product names and amounts, with natural-language wording and synonyms.

##### 1. Approximate nearest-neighbor vector search with HNSW

```python
vector_search = VectorSearch(
    algorithms=[
        HnswAlgorithmConfiguration(
            name="hnsw-config",
            parameters=HnswParameters(
                m=12,
                ef_construction=300,
                ef_search=130,
                metric="cosine",
            ),
        ),
    ],
    profiles=[
        VectorSearchProfile(
            name="insurance-vector-profile",
            algorithm_configuration_name="hnsw-config",
        ),
    ],
)
```

`HnswAlgorithmConfiguration` selects the HNSW (Hierarchical Navigable Small World) algorithm. HNSW builds a graph of related embedding vectors so Azure AI Search can find approximate nearest neighbours quickly without comparing the query vector with every indexed chunk.

The `VectorSearchProfile` separates the reusable algorithm configuration from the field binding. The schema connects `content_vector` to `insurance-vector-profile`; the profile then selects `hnsw-config`. This keeps the field declaration concise and permits another vector field to reuse the same algorithm configuration later.

| Setting | Design purpose | Trade-off |
|---|---|---|
| `metric="cosine"` | Compare the direction of embedding vectors, which is a standard similarity measure for text embeddings. | It assumes cosine similarity is appropriate for the embedding model; changing the model may require revisiting the metric. |
| `m=12` | Controls approximately how many graph connections each indexed vector has. | A moderate value limits index memory and build cost, but lower connectivity can slightly reduce recall compared with a larger `m`. |
| `ef_construction=300` | Searches a broad candidate set while building the graph, improving graph quality. | Increases indexing time and resource use. This is usually acceptable for a document index built less often than it is queried. |
| `ef_search=130` | Searches more graph candidates at query time, increasing the chance of finding strong neighbours. | Improves recall but adds query latency and compute cost. |
| `k_nearest_neighbors` at query time | Limits how many vector-nearest chunks enter the candidate set. | A smaller value is faster but may exclude useful evidence; a larger value improves recall but can introduce more weak candidates. |

The configuration favors answer quality over the cheapest possible retrieval: a relatively high construction breadth (`300`) and query breadth (`130`) aim to avoid missing relevant policy text. The cost is a larger/slower vector index and more query work. These values should be benchmarked using real insurance questions, a target latency, and retrieval-quality measures such as recall@k or grounded-answer quality.

##### 2. Semantic ranking configuration

```python
semantic_config = SemanticConfiguration(
    name="insurance-semantic-config",
    prioritized_fields=SemanticPrioritizedFields(
        title_field=SemanticField(field_name="title"),
        content_fields=[SemanticField(field_name="content")],
        keywords_fields=[SemanticField(field_name="document_type")],
    ),
)
semantic_search = SemanticSearch(configurations=[semantic_config])
```

The semantic configuration tells Azure AI Search which fields represent the document's meaning when it semantically reranks candidates and creates extractive captions:

- `title` is the primary document label, useful for policy or product names.
- `content` is the evidence the model should evaluate for relevance.
- `document_type` supplies a short category signal, such as text, table, or image-derived content.

This is an intentional field-prioritization design. A policy title may identify the correct product, while `content` contains the coverage conditions and exclusions needed to answer the question. `document_type` can help distinguish a normal policy paragraph from enriched content such as a table description.

The trade-off is that semantic ranking adds latency and service cost after initial retrieval. It also depends on clean, well-populated titles and content. If titles are poor, content is noisy, or document types are inconsistent, the reranker has weaker signals and may confidently rank the wrong chunk. The indexing code's text cleaning and metadata extraction are therefore part of retrieval quality, not merely data preparation.

##### 3. One index serving multiple retrieval stages

```python
index = SearchIndex(
    name=index_name,
    fields=fields,
    vector_search=vector_search,
    semantic_search=semantic_search,
)
```

The final `SearchIndex` holds the field schema, vector-search configuration, and semantic-search configuration together. The retriever can then make a hybrid request that uses `search_text`, `content_vector`, and `insurance-semantic-config` in one search operation.

This centralizes retrieval behavior in the index definition, improving consistency between ingestion and querying. The trade-off is operational coupling: schema changes, embedding-model changes, or major HNSW parameter changes typically require reindexing and careful rollout because the indexed vectors and query configuration must remain compatible.

#### 3. Batch embedding and upload (`src/indexing_rag.py:L317-L336, L338-L362`)
```python
def index_documents(documents: List[Dict], batch_size: int, language: str, embeddings: AzureOpenAIEmbeddings) -> None:
    """Index documents in batches, handle failures without stopping."""
    total = len(documents)
    for i in range(0, total, batch_size):
        batch = documents[i:i + batch_size]
        for doc in batch:
            doc['content_vector'] = get_embedding(doc['content'], embeddings)
        batch_ids = [doc.get('id') for doc in batch]
        try:
            search_client.upload_documents(documents=batch)
            processed = min(i + batch_size, total)
            logger.info(f"Indexed {processed}/{total} documents.")
        except Exception as e:
            logger.error(f"[Indexer] Failed to upload batch {i//batch_size+1}: {e}", exc_info=True)
            for doc in batch:
                try:
                    search_client.upload_documents(documents=[doc])
                except Exception as doc_e:
                    logger.error(f"[Indexer] Failed to upload doc ID {doc.get('id')}: {doc_e}")
    logger.info(f"✓ Successfully indexed all {total} new documents.")
# ...
def process_document(documents: List[Dict[str, Any]], index_name: str, recreate_index: bool, batch_size: int, language: str, embeddings: AzureOpenAIEmbeddings) -> None:
    """Main document processor for Azure Search vector+semantic index."""
    if not documents:
        logger.warning("No documents found, aborting index build.")
        return
    index_exists = True
    try:
        index_client.get_index(index_name)
        logger.info(f"Index '{index_name}' exists.")
    except Exception:
        index_exists = False
    if recreate_index and index_exists:
        try:
            logger.info(f"Recreating index: {index_name}")
            index_client.delete_index(index_name)
            index_exists = False
            logger.info(f"Deleted existing index '{index_name}'.")
        except Exception as e:
            logger.error(f"Failed to delete index '{index_name}': {e}")
    if not index_exists:
        logger.info(f"Creating index: {index_name} ...")
        create_index(index_name)
    else:
        logger.info(f"Reusing index: {index_name} (no creation).")
    index_documents(documents, batch_size, language, embeddings)
```

**Why it matters**

The code embeds each document, uploads in batches, and falls back to per-document upload if a batch fails. That is a practical production detail: ingestion keeps moving even when one batch is problematic.

**Overall script flow**

1. Prepare token-usage logging and Azure Search clients.
2. Clean text and transform chunk structures into index documents.
3. Create or recreate the Azure Search index schema.
4. Embed chunks and upload them with batch-level fallback handling.


### `src/document_intelligence_rag.py`

This file exists to **enrich raw PDFs before indexing**. It has used Azure Document Intelligence plus LLM summarization so tables and figures become retrievable text instead of dead visual artifacts.

**Three most important code snippets**

#### 1. Layout analysis from blob storage (`src/document_intelligence_rag.py:L48-L72`)
```python
def analyze_document_from_blob(blob_client: BlobClient, azure_client: DocumentIntelligenceClient) -> Tuple[Optional[AnalyzeResult], Optional[str]]:
    """Analyze PDF from Azure Blob; log and handle all exceptions. Never returns uncaught errors."""
    print(f"\nAnalyzing document: {blob_client.blob_name}")
    blob_url = blob_client.url
    result_id = None
    result = None
    try:
        poller = azure_client.begin_analyze_document(
            model_id="prebuilt-layout",
            body={"urlSource": blob_url},
            output=["figures"]
        )
        result = poller.result()
        result_id = poller.details["operation_id"]
        print(f"✓ Analysis complete: {len(result.pages)} pages")
        return result, result_id
    except Exception as e:
        logger.error(f"[DocIntel] Document analysis failed: {e}", exc_info=True)
        record = {"filename": blob_client.blob_name.split("/")[-1], "issue": str(e)}
        try:
            with open("file_with_issues", "a", encoding="utf-8") as f:
                f.write(json.dumps(record, ensure_ascii=False) + "\n")
        except Exception as file_log_exc:
            logger.warning(f"[DocIntel] Could not log issue to file: {file_log_exc}")
        return result, result_id
```

**Why it matters**

`analyze_document_from_blob(...)` calls the `prebuilt-layout` model directly against Azure Blob URLs and requests figure output. That establishes the structured page representation used by the rest of the ingestion pipeline.

#### 2. Table extraction plus LLM verbalization (`src/document_intelligence_rag.py:L84-L95, L96-L107, L133-L150`)
```python
        current_prompt= (
            "You are an assistant that summarizes and describes a table from insurance documents.\n"
            "Rules:\n"
            "1. Identify the main topic or category (e.g., policy details, claims, financial figures).\n"
            '2. Highlight important values (e.g., numbers, percentages, dates) and focus on key columns such as "Coverage," "Reimbursements," or "Limits" if present.\n'
            "3. Write a short summary (max 100 words) using the information above to support document search and retrieval.\n"
            "Be concise, factual, and neutral. Avoid speculation or unnecessary details."
        )
        user_message = (
            "Analyze this table from an insurance document and provide a detailed description.\n\n"
            "Here is the table:\n" + markdown_table
        )
# ...
        messages = [
            SystemMessage(content=current_prompt),
            HumanMessage(content=user_message),
        ]
        response = llm.invoke(messages)
        token_table={}
        token_table["type"] = "table_verbalization" 
        token_table["input_tokens"] = (response.usage_metadata).get("input_tokens", 0)  # type: ignore
        token_table["output_tokens"] = (response.usage_metadata).get("output_tokens", 0)  # type: ignore
        token_table["total_tokens"] = (response.usage_metadata).get("total_tokens", 0)  # type: ignore
        token_chat_usage(token_verb_table, token_table, user_message)
        return response.content.strip() if response else "" # type: ignore
# ...
    for idx, table in enumerate(result.tables): #type: ignore
        if table.bounding_regions and table.bounding_regions[0].page_number == page_number:
            markdown_chunks = _table_to_markdown(table)
            for chunk_idx, markdown_table in enumerate(markdown_chunks):
                verbalized_table = verbalize_table(markdown_table, llm)
                table_info = {
                    'id': f"table_{idx+1}_chunk_{chunk_idx+1}",
                    'row_count': table.row_count,
                    'column_count': table.column_count,
                    'markdown': markdown_table,
                    'content': verbalized_table if verbalized_table else "",
                    'focus_category': "insurance/reimbursement",
                    'important_columns': ["Coverage", "Reimbursement", "Limits"],
                    'page_number': page_number,
                    'chunk_number': chunk_idx + 1,
                    'total_chunks': len(markdown_chunks),
                }
                tables.append(table_info)
```

**Why it matters**

This combined path converts tables to markdown, sends them through an LLM prompt focused on insurance-relevant details, and stores both structure and textual description. That is a strong RAG design choice because retrieval works best on language, not raw cell grids.

#### 3. Figure extraction and vision verbalization (`src/document_intelligence_rag.py:L166-L183, L203-L212, L214-L228`)
```python
    for idx, figure in enumerate(result.figures): # type: ignore
        if figure.bounding_regions and figure.bounding_regions[0].page_number == page_number:
            try:
                caption = figure.caption.content if figure.caption else ""
                region = figure.bounding_regions[0]
                polygon = region.polygon
                width = max(polygon[0::2]) - min(polygon[0::2])
                height = max(polygon[1::2]) - min(polygon[1::2])
                area = width * height
                figure_id = figure.id if figure.id is not None else f"figure_{idx}"
                image_bytes = azure_client.get_analyze_result_figure("prebuilt-layout", result_id, figure_id)
                if area <= 2.0:
                    logos_extracted.append({
                        'caption': caption if caption else "logo",
                        'image_bytes': image_bytes
                    })
                else:
                    extracted_images.append({
# ...
        image_b64 = base64.b64encode(image_bytes).decode('utf-8')
        prompt = (
            "You are an assistant that helps summarize and describe image from insurance documents.\n"
            "Rules:\n"
            "1. Identify the type of visual in the image (e.g., chart, table, diagram, or other) and the main topic or category it represents.\n"
            "2. Highlight important values (e.g., numbers, percentages, dates, or other relevant values), and briefly explain what the visual shows.\n"
            "3. If possible, relate the visual to insurance topics using the provided context: " + (content if content else "") + "\n"
            "4. Provide a short summary (max 100 words) using the information above to support document search and retrieval.\n"
            "Be clear, neutral, and factual. Avoid speculation or unnecessary details."
        )
# ...
        messages = [
            SystemMessage(content=prompt),
            HumanMessage(content=[
                {"type": "text", "text": user_message},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}", "detail": "high"}}
            ])
        ]
        response = llm.invoke(messages)
        token_image={}
        token_image["type"] = "image_verbalization" 
        token_image["input_tokens"] = (response.usage_metadata).get("input_tokens", 0)  # type: ignore
        token_image["output_tokens"] = (response.usage_metadata).get("output_tokens", 0)  # type: ignore
        token_image["total_tokens"] = (response.usage_metadata).get("total_tokens", 0)  # type: ignore
        token_chat_usage(token_verb_image, token_image, user_message)
        return response.content.strip() if response else "" # type: ignore
```

**Why it matters**

The code separates small logo-like figures from larger images, then turns the larger visuals into text with a vision-capable prompt. This is how the pipeline preserves meaning from images instead of ignoring them during indexing.

**Overall script flow**

1. Analyze a PDF layout from Blob Storage.
2. Resolve source-document URLs.
3. Turn tables into markdown and then into short retrieval-friendly descriptions.
4. Extract figures, distinguish logos from substantial visuals, and verbalize meaningful images.
5. Return text-ready metadata for the indexing stage.


### `src/initialization_rag.py`

This is the **top-level ingestion orchestrator**: it glues together Blob Storage, Document Intelligence, chunk caching, text splitting, image/table enrichment, and final indexing.

**Three most important code snippets**

#### 1. Embedding/chat clients and chunking policy (`src/initialization_rag.py:L58-L77`)
```python
llm_embedding = AzureOpenAIEmbeddings(
    azure_endpoint=InsuranceConfig.AZURE_OPENAI_EMBEDDINGS_ENDPOINT,
    api_key=InsuranceConfig.AZURE_OPENAI_EMBEDDINGS_KEY,# type: ignore
    api_version=InsuranceConfig.AZURE_OPENAI_EMBEDDINGS_API_VERSION,
    azure_deployment=InsuranceConfig.AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT,
    model=InsuranceConfig.AZURE_OPENAI_EMBEDDINGS_MODEL
)
llm_chat = AzureChatOpenAI(
    azure_endpoint=InsuranceConfig.AZURE_OPENAI_ENDPOINT,
    api_key=InsuranceConfig.AZURE_OPENAI_API_KEY,# type: ignore
    api_version=InsuranceConfig.AZURE_OPENAI_API_VERSION,
    azure_deployment=InsuranceConfig.AZURE_OPENAI_DEPLOYMENT,
    model=InsuranceConfig.AZURE_OPENAI_DEPLOYMENT
)
```

**separators=["\n\n", "\n", " ", ""]**: 

Tells the splitter to try splitting by **paragraphs (\n\n)** first, then **sentences (\n)**, then **words ( )**, and finally **individual characters ("")** if a chunk is still too large.

```python
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=240,
    separators=["\n\n", "\n", " ", ""]
)
```

**Why it matters**

This block establishes the two model clients used during ingestion and fixes the chunking strategy at 1200 characters with 240 overlap. This is where balanced context preservation against index granularity.

#### 2. Reusing cached chunks (`src/initialization_rag.py:L79-L103`)
```python
def load_index_chunks_from_directory(index_name: str, recreate_index: bool, batch_size: int) -> str:
    """Load .pkl chunk files and index metadata, robust file handling."""
    script_dir = os.path.dirname(os.path.abspath(__file__))
    chunks_dir = os.path.join(script_dir, "azuredocumentintelligence")
    os.makedirs(chunks_dir, exist_ok=True)

    if os.path.isdir(chunks_dir):
        all_chunks = []
        for f in sorted(os.listdir(chunks_dir)):
            chunk_path = os.path.join(chunks_dir, f)
            try:
                with open(chunk_path, "rb") as chunk_file:
                    chunk = pickle.load(chunk_file)
                    all_chunks.extend(chunk)
            except Exception as e:
                logger.warning(f"Failed to load chunk file {chunk_path}: {e}")
        print(f"Total chunks loaded from {chunks_dir}: {len(all_chunks)}")
        documents = doc_split_pdf(all_chunks, "FR")
        print(f"Total documents after created the appropriate metadata: {len(documents)}")
        process_document(documents, index_name, recreate_index, batch_size, "FR", llm_embedding)
        print(f"\nTotal chunks indexed: {len(documents)}")
        return chunks_dir
    else:
        logger.warning(f"No chunk directory found. New chunks will be created.")
        return chunks_dir
```

**Why it matters**

`load_index_chunks_from_directory(...)` lets the pipeline resume from saved `.pkl` chunks instead of forcing a full re-analysis every time. That is an operational optimization that matters when document analysis is expensive.

#### 3. End-to-end build loop (`src/initialization_rag.py:L124-L138, L146-L152, L171-L180, L201-L218, L225-L235`)
```python
        for blob in container_client.list_blobs(name_starts_with=folder_prefix):
            blob_name = blob.name
            blob_client = blob_service_client.get_blob_client(
                container=storage_container_name,
                blob=blob_name
            )
            print(f"\nProcessing document: {blob_name}")
            try:
                pdf_bytes = blob_client.download_blob().readall()
                result, result_id = analyze_document_from_blob(blob_client, doc_intelligence_client)
            except Exception as e:
                logger.error(f"Failed to download/analyze blob {blob_name}: {e}")
                continue
            if not result:
                print(f"⚠️ Document analysis failed for {blob_name}. Skipping this document.")
# ...
                    cleaned_lines, title, edition_date = clean_pdf_text(page, pdf_bytes)
                    if not cleaned_lines or str(cleaned_lines).strip() == "":
                        print(f"⚠️ Blank or unreadable page at page {page_num} in {blob_name}. Skipping text chunking.")
                        continue

                    url_filename = find_url_by_filename(file_name)
                    text_chunks = text_splitter.split_text(cleaned_lines)
# ...
                    if result_id is None:
                        print(f"Error: Cannot process images because result_id is None for page {page_num}.")
                    else:
                        images, logos = extract_images_from_result(
                            result, blob_name, pdf_bytes, result_id, doc_intelligence_client, page_num
                        )
                        if images:
                            for img in images:
                                ref_content = text_chunks[0] if text_chunks else ""
                                description = verbalize_image_from_bytes(img, ref_content, llm_chat)
# ...
                    tables = extract_tables_from_result(result, page_num, llm_chat)
                    if tables:
                        for table in tables:
                            table_metadata = {
                                'id': table['id'],
                                'type': "table",
                                'title': title,
                                'filename': file_name,
                                'page_number': page_num,
                                'url': url_filename,
                                'content': table['markdown'],
                                'description': table['content'],
                                'focus_category': table.get('focus_category', ""),
                                'important_columns': table.get('important_columns', []),
                                'chunk_number': table.get('chunk_number', None),
                                'total_chunks': table.get('total_chunks', 1),
                            }
                            chunk.append(table_metadata)
# ...
            print(f"File: {blob_name.split('/')[-1]} — Chunks created: {len(chunk)}")
            chunk_filename = f"chunk_{blob_name.split('/')[-1]}.pkl"
            chunk_path = os.path.join(chunks_dir, chunk_filename)
            try:
                with open(chunk_path, "wb") as f:
                    pickle.dump(chunk, f)
            except Exception as e:
                logger.error(f"Failed to write chunk file {chunk_path}: {e}")
        try:
            documents = doc_split_pdf(chunk, lang_code)
            process_document(documents, index_name, recreate_index, batch_size, lang_code, llm_embedding)
```

**Why it matters**

`build_rag_index(...)` downloads PDFs, analyzes pages, splits text, verbalizes images/tables, serializes chunk files, and finally calls `process_document(...)`. This is the actual assembly line that turns source PDFs into a searchable knowledge base.

**Overall script flow**

1. Initialize storage, Document Intelligence, embeddings, and chat clients.
2. Try to reuse locally cached chunk files.
3. If no cache exists, iterate through blobs, page by page.
4. Clean/split text, enrich images and tables, and store intermediate chunk pickles.
5. Convert the accumulated chunk data into Azure Search documents and index them.


## Group 5 — `frontend`

Files mapped here:

- `src/frontend.py`
- `src/requirements.txt`


### `src/frontend.py`

This is the **user-facing Streamlit application**. Its job is to keep the user journey coherent: create a backend session, guide language/policy selection, submit chat messages, surface governance outcomes, and let the user review history and rate responses.

**Three most important code snippets**

#### 1. Session bootstrap (`src/frontend.py:L222-L257`)
```python
def initialize_session_state():
    if "user_info" not in st.session_state:
        st.session_state["user_info"] = authenticate_user()

    user_info = st.session_state["user_info"]

    if "session_data" not in st.session_state:
        session_data = create_session(user_info["user_id"])

        if not session_data:
            language = st.session_state.get("language", "en")
            st.error(language_manager.get_message("error", language))
            logger.error("Session initialization returned empty session_data")
            st.stop()

        st.session_state["session_data"] = session_data
        st.session_state["session_id"] = session_data["session_id"]
        st.session_state["user_id"] = session_data["user_id"]
        st.session_state["username"] = user_info["username"]
        st.session_state["default_language"] = InsuranceConfig.DEFAULT_LANGUAGE
        st.session_state["language"] = session_data["language"]
        if "language_confirmed" not in st.session_state:
            st.session_state["language_confirmed"] = False
        st.session_state["policies"] = session_data["policies"]
        st.session_state["past_sessions"] = session_data["past_sessions"]
        st.session_state["policy_type"] = session_data.get("policy_type")
        pending_info = get_pending_query(session_data["session_id"])
        st.session_state["pending_query"] = pending_info.get("pending_query")
        st.session_state["language_confirmed"] = pending_info.get(
            "language_confirmed", st.session_state.get("language_confirmed", False)
        )
        if pending_info.get("language"):
            st.session_state["language"] = pending_info.get("language")
        if pending_info.get("policy_type") is not None:
            st.session_state["policy_type"] = pending_info.get("policy_type")
```

**Why it matters**

`initialize_session_state()` is the backbone of the UI. It creates the backend session, stores user/session/policy/history metadata in `st.session_state`, and reloads pending-query information so the UX can recover mid-flow.

#### 2. Governance/operator control panel (`src/frontend.py:L374-L378, L379-L389, L394-L409, L423-L428`)
```python
    st.caption(language_manager.get_message("governance_help", language))
    if settings.get("azure_content_safety_configured"):
        st.success(language_manager.get_message("governance_azure_ok", language))
    else:
        st.warning(language_manager.get_message("governance_azure_missing", language))
# ...
    check_input = st.checkbox(
        language_manager.get_message("governance_check_input", language),
        value=bool(settings.get("check_input", True)),
        key="gov_check_input",
    )
    check_output = st.checkbox(
        language_manager.get_message("governance_check_output", language),
        value=bool(settings.get("check_output", True)),
        key="gov_check_output",
    )
    enable_content_safety = st.checkbox(
# ...
    enable_jailbreak = st.checkbox(
        language_manager.get_message("governance_jailbreak", language),
        value=bool(settings.get("enable_jailbreak_detection", True)),
        key="gov_jailbreak",
    )
    enable_azure = st.checkbox(
        language_manager.get_message("governance_azure_cs", language),
        value=bool(settings.get("enable_azure_content_safety", True)),
        key="gov_azure_cs",
    )
    enable_pii = st.checkbox(
        language_manager.get_message("governance_pii", language),
        value=bool(settings.get("enable_pii", True)),
        key="gov_pii",
    )
    if st.button(language_manager.get_message("governance_save", language), use_container_width=True, key="gov_save"):
# ...
    audit_entries = get_governance_audit(limit=6)
    if audit_entries:
        st.caption("Recent checks: " + " | ".join(
            f"{entry.get('action', '')}: {entry.get('result', '')}"
            for entry in reversed(audit_entries[-6:])
        ))
```

**Why it matters**

This block shows the frontend is not just a chat box. It exposes runtime governance toggles and recent audit feedback, which means the UI doubles as a lightweight operations surface for the live guardrails.

#### 3. Message handling and deferred replay (`src/frontend.py:L546-L555, L556-L568, L585-L595, L597-L603, L618-L624`)
```python
def process_pending_query(pending_query: str):
    language = st.session_state.get("language", "en")
    with st.spinner(language_manager.get_message("processing_query", language)): #customized language
        response = send_message(
            session_id=st.session_state["session_id"],
            message=pending_query,
            policy_type=st.session_state.get("policy_type"),
            policies=st.session_state.get("policies")
        )
    if response:
# ...
        assistant_message = response["message"]
        governance = response.get("governance") or {}
        st.session_state["messages"].append({
            "role": "assistant",
            "content": assistant_message,
            "governance": governance,
        })
        reset_logout_timer()
        st.session_state["chat_input_disabled"] = False
        needs_clarification = response.get("needs_clarification", "")
        if needs_clarification != "unclear" and not governance.get("blocked"):
            st.session_state["pending_evaluation"] = True
        st.session_state["pending_query"] = None
# ...
    if not st.session_state.get("language_confirmed"):
        st.session_state["pending_query"] = user_message
        system_message = language_manager.get_message("language_confirmation", language)
        st.session_state["messages"].append({
            "role": "assistant",
            "content": system_message
        })
        with st.chat_message("assistant"):
            st.markdown(system_message)
        st.session_state["chat_input_disabled"] = False
        st.rerun()
# ...
    with st.chat_message("assistant"):
        with st.spinner(language_manager.get_message("thinking", language)): #customized language
            response = send_message(
                session_id=st.session_state["session_id"],
                message=user_message,
                policy_type=st.session_state.get("policy_type"),
                policies=st.session_state.get("policies")
# ...
            if needs_clarification == "unclear":
                st.session_state["pending_query"] = user_message
            else:
                st.session_state["pending_query"] = None
                if not governance.get("blocked"):
                    st.session_state["pending_evaluation"] = True
                st.session_state["policy_type"] = None
```

**Why it matters**

`process_pending_query(...)` and `handle_user_message(...)` implement the chat state machine: queue a message if prerequisites are missing, replay it later, render governance status, and enable post-answer evaluation only when the answer was not blocked. That is the clearest expression of frontend/business-logic coupling in the UI.

**Overall script flow**

1. Configure Streamlit and logging.
2. Authenticate the user and initialize session state.
3. Render a sidebar for language, policies, governance settings, and history.
4. Render the active chat or a historical session view.
5. Submit chat messages to the backend, preserve pending queries when needed, and surface governance/evaluation feedback in the UI.

**Note on snippets**

I intentionally avoided reproducing the hardcoded login constants from the file in this notebook, because duplicating them would create a new copy of sensitive material. The login flow still belongs to the script's overall design, but it is safer to describe than to quote.


### `src/requirements.txt` in the `frontend` group

The production layout copies the **same** `src/requirements.txt` file into the frontend package as well.

**Why it belongs here**

Even though the Streamlit app is only one module, the layout script still gives it the shared Python dependency manifest. That is evidence that the frontend and backend are deployed from a coordinated runtime stack rather than from two totally separate environments.

#### 1. UI/runtime dependencies (`src/requirements.txt:L41-L45 and L167-L168`, decoded from UTF-16)
```text
fastapi==0.133.0
filelock==3.25.0
frozenlist==1.8.0
fsspec==2026.2.0
gitdb==4.0.12
```

```text
streamlit==1.55.0
tabulate==0.9.0
```

**Why it matters**

This excerpt shows `fastapi` and `streamlit` coexisting in the same dependency set, which matches the repo's two-application architecture.

#### 2. HTTP/integration dependencies (`src/requirements.txt:L151-L156`, decoded from UTF-16)
```text
redis==5.0.8
referencing==0.37.0
regex==2026.1.15
requests==2.32.5
requests-oauthlib==2.0.0
requests-toolbelt==1.0.0
```

**Why it matters**

These packages support the frontend/backend/API integration story (`redis`, `requests`, and OAuth-related libraries), reinforcing that the frontend is an API client rather than a fully standalone application.

**Overall file logic**

The frontend does not have its own bespoke dependency file in `src/`; it reuses the project's shared environment definition.
